Lets Load Data 

In [2]:
import pandas as pd 
import numpy as np 
import os
from pathlib import Path

In [3]:
dataset_dir = Path("C:/Users/vaidi/.cache/kagglehub/competitions/enveda-CASMI26-molecule-id-mass-spectra")

print("Files in dataset:", os.listdir(dataset_dir))

file_path = dataset_dir / 'train.parquet'

df_tr = pd.read_parquet(file_path)
df_tr.head()

Files in dataset: ['sample_submission.csv', 'test.parquet', 'train.parquet']


,ingest_lib,normalized_smiles,inchikey,inchikey14,molecular_formula,ionization_mode,instrument_type,adduct,adduct_orig,precursor_mz,precursor_error_ppm,ms2_mzs,ms2_normalized_intensities,num_peaks,base_peak_intensity,collision_energy_ev,collision_energy_orig,collision_energy_orig_units
0,drug_plus,O=C1NC(=O)c2cc(Nc3ccccc3)c(Nc3ccccc3)cc21,AAALVYBICLMAMA-UHFFFAOYSA-N,AAALVYBICLMAMA,C20H15N3O2,positive,NaN,[M+H]+,[M+H]+,330.1237,1.671398,"[92.0495, 93.0573, 94.0607, 116.107, 123.1168,...","[0.0171037726229424, 0.328091214993177, 0.0107...",52,NaN,None,NaN,unknown
1,drug_plus,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1,AAKJLRGGTJKAMG-UHFFFAOYSA-N,AAKJLRGGTJKAMG,C22H23N3O4,positive,NaN,[M+K]+,[M+K]+,432.1320,1.302193,"[82.1451, 89.0599, 158.9638, 248.0836, 250.098...","[0.00144055962319783, 0.959302375462881, 0.5, ...",28,NaN,None,NaN,unknown
2,drug_plus,CC1(C)CCC(C)(C)c2cc(C(O)C(O)=Nc3ccc(C(=O)O)cc3...,AANFHDFOMFRLLR-UHFFFAOYSA-N,AANFHDFOMFRLLR,C23H26FNO4,positive,NaN,[M+Na]+,[M+Na]+,422.1738,1.316437,"[91.0539, 111.117, 131.0856, 150.035, 171.0803...","[0.00262159533923045, 0.0184752871374318, 0.00...",41,NaN,None,NaN,unknown
3,drug_plus,CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21,AAOVKJBEBIDNHE-UHFFFAOYSA-N,AAOVKJBEBIDNHE,C16H13ClN2O,positive,NaN,[M+H]+,[M+H]+,285.0789,1.984579,"[58.0287, 65.0386, 77.0386, 89.0386, 90.0464, ...","[0.00908241772391234, 0.00183110584152731, 0.0...",166,NaN,None,NaN,unknown
4,drug_plus,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,ACFIXJIJDZMPPO-UHFFFAOYSA-N,ACFIXJIJDZMPPO,C21H30N7O17P3,positive,NaN,[M+H]+,[M+H]+,746.0984,0.708429,"[53.622, 60.372, 62.949, 74.853, 79.885, 91.29...","[0.2838, 0.11773, 0.15448, 0.11773, 0.26637, 0...",186,NaN,None,NaN,unknown


In [4]:
df_tr.columns

Index(['ingest_lib', 'normalized_smiles', 'inchikey', 'inchikey14',
       'molecular_formula', 'ionization_mode', 'instrument_type', 'adduct',
       'adduct_orig', 'precursor_mz', 'precursor_error_ppm', 'ms2_mzs',
       'ms2_normalized_intensities', 'num_peaks', 'base_peak_intensity',
       'collision_energy_ev', 'collision_energy_orig',
       'collision_energy_orig_units'],
      dtype='str')

In [5]:
file_path = dataset_dir / 'test.parquet'
df_ts = pd.read_parquet(file_path)

df_ts.columns

Index(['molecule_id', 'spectrum_id', 'ms2_mzs', 'ms2_normalized_intensities',
       'base_peak_intensity', 'adduct', 'ionization_mode', 'instrument_type',
       'precursor_mz', 'collision_energy_orig', 'collision_energy_ev',
       'collision_energy_orig_units'],
      dtype='str')

In [6]:
df_tr.columns

Index(['ingest_lib', 'normalized_smiles', 'inchikey', 'inchikey14',
       'molecular_formula', 'ionization_mode', 'instrument_type', 'adduct',
       'adduct_orig', 'precursor_mz', 'precursor_error_ppm', 'ms2_mzs',
       'ms2_normalized_intensities', 'num_peaks', 'base_peak_intensity',
       'collision_energy_ev', 'collision_energy_orig',
       'collision_energy_orig_units'],
      dtype='str')

**Feature Engineering**

**Noise Reduction:**

We are filtering the spectrum and removing the noise

In [7]:
def filter_spectrum_advanced(mzs, intensities, precursor_mz, threshold=0.01, top_n=128):
    mzs = np.asarray(mzs, dtype=np.float32)
    intensities = np.asarray(intensities, dtype=np.float32)

    # 1. Intensity threshold
    mask = intensities >= threshold             #[true , false , true ,false,....]
    
    # 2. Precursor limit (Physics rule: +2.0 Da allowance for isotopes)
    mask = mask & (mzs <= (precursor_mz + 2.0)) # its like the fragments should not have m/z greater than original structure(+2 is for isotopes edge case)

    mzs = mzs[mask]
    intensities = intensities[mask]

    # 3. Keep only the Top-N most intense peaks
    if len(intensities) > top_n:
        top_indices = np.argsort(intensities)[-top_n:]
        mzs = mzs[top_indices]
        intensities = intensities[top_indices]

    return mzs.tolist(), intensities.tolist()

# Apply the updated filter (Note: we pass in the precursor_mz now)
filtered = df_tr.apply(
    lambda row: filter_spectrum_advanced(
        row["ms2_mzs"], 
        row["ms2_normalized_intensities"], 
        row["precursor_mz"]
    ), axis=1
)

df_tr["ms2_mzs"] = filtered.str[0]
df_tr["ms2_normalized_intensities"] = filtered.str[1]

**Sparse Bining CSR : It will convert messy variable_length lists into uniform , memory efficient matrix**

We are gonna use CSR matrix format to handle the multiple m/z and intensities 

In [8]:
import numpy as np
from scipy.sparse import csr_matrix
import gc

def create_sparse_binned_matrix(df, bin_size=0.1, max_mz=1000.0):
    # 1000 / 0.1 = 10,000 bins
    num_bins = int(max_mz / bin_size)
    
    # 1. Count peaks per spectrum to map them back later
    peak_counts = df['ms2_mzs'].str.len().to_numpy()
    
    # 2. Flatten everything to C-speed 1D arrays
    all_mzs = np.concatenate(df['ms2_mzs'].to_numpy())
    all_ints = np.concatenate(df['ms2_normalized_intensities'].to_numpy())
    
    # 3. Create row indices (e.g., [0, 0, 1, 2, 2, 2...])
    row_indices = np.repeat(np.arange(len(df)), peak_counts)
    
    # 4. Map m/z to column indices
    col_indices = (all_mzs / bin_size).astype(np.int32)
    
    # 5. Filter out peaks that exceed max_mz
    valid_mask = (col_indices >= 0) & (col_indices < num_bins)
    
    filtered_rows = row_indices[valid_mask]
    filtered_cols = col_indices[valid_mask]
    filtered_vals = all_ints[valid_mask]
    
    # Free up memory explicitly before building the matrix
    del all_mzs, all_ints, row_indices, col_indices, valid_mask
    gc.collect()
    
    # 6. Build the CSR matrix
    sparse_matrix = csr_matrix(
        (filtered_vals, (filtered_rows, filtered_cols)),
        shape=(len(df), num_bins),
        dtype=np.float32
    )
    
    return sparse_matrix

print("Vectorizing spectra...")
X_train_sparse = create_sparse_binned_matrix(df_tr, bin_size=0.1, max_mz=1000.0)

print(f"Matrix shape: {X_train_sparse.shape} (Rows: Spectra, Cols: 0.1 Da Bins)")
print(f"Memory footprint: {X_train_sparse.data.nbytes / 1e6:.2f} MB")

Vectorizing spectra...
Matrix shape: (2539608, 10000) (Rows: Spectra, Cols: 0.1 Da Bins)
Memory footprint: 191.38 MB


**Intensity Scaling**

This ensures that the lower intensity m/z dont get forgotten by the neural network

In [9]:
from sklearn.preprocessing import normalize

import numpy as np

X_train_sparse.data = np.sqrt(X_train_sparse.data)

X_train_nn_ready = normalize(X_train_sparse, norm="max",axis=1)

print(f"Final NN Matrix Shape: {X_train_nn_ready.shape}")
print(f"Max intensity is now: {X_train_nn_ready.max()}")

Final NN Matrix Shape: (2539608, 10000)
Max intensity is now: 1.0


We are gonna tackle the collision enery column and then do the scaling and encoding


In [10]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

def extract_ce_stats(ce_series):
    """Fast extraction of statistical features from variable-length lists."""
    mins, maxs, means, medians, counts = [], [], [], [], []
    
    for x in ce_series:
        # Check if valid list/array with elements
        if isinstance(x, (list, np.ndarray)) and len(x) > 0:
            mins.append(float(np.min(x)))
            maxs.append(float(np.max(x)))
            means.append(float(np.mean(x)))
            medians.append(float(np.median(x)))
            counts.append(len(x))
        else:
            # Fallback for missing data
            mins.append(20.0)
            maxs.append(20.0)
            means.append(20.0)
            medians.append(20.0)
            counts.append(0)
            
    return mins, maxs, means, medians, counts



In [ ]:
def extract_features_(df, is_train=True, preprocessor=None):
    print("Extracting advanced tabular metadata...")
    
    # 1. Extract the 5 collision energy features
    ce_mins, ce_maxs, ce_means, ce_medians, ce_counts = extract_ce_stats(df['collision_energy_ev'])
    
    # 2. Build the dense DataFrame
    meta_df = pd.DataFrame({
        'precursor_mz': df['precursor_mz'].fillna(0),
        'base_peak_intensity': df['base_peak_intensity'].fillna(0),
        'ce_min': ce_mins,
        'ce_max': ce_maxs,
        'ce_mean': ce_means,
        'ce_median': ce_medians,
        'ce_count': ce_counts,
        'adduct': df['adduct'].fillna('unknown'),
        'ionization_mode': df['ionization_mode'].fillna('unknown')
    })
    
    # 3. Define the numerical columns we want to scale
    numeric_cols = [
        'precursor_mz', 'base_peak_intensity', 
        'ce_min', 'ce_max', 'ce_mean', 'ce_median', 'ce_count'
    ]
    
    # 4. Scale numbers and One-Hot Encode categories
    if is_train:
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', StandardScaler(), numeric_cols),
                ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['adduct', 'ionization_mode'])
            ]
        )
        meta_features = preprocessor.fit_transform(meta_df)
        return meta_features.astype(np.float32), preprocessor
    else:
        meta_features = preprocessor.transform(meta_df)
        return meta_features.astype(np.float32)

# --- Execution ---
X_train_meta, meta_preprocessor = extract_metadata_features_advanced(df_tr, is_train=True)

print(f"Metadata Matrix Shape: {X_train_meta.shape}")

Extracting advanced tabular metadata...
Metadata Matrix Shape: (2539608, 130)
